# RAG Ablation Analysis — Institutional 6-Case Evaluation

Reproduces the paper's Table 2, Table 2b, and the associated delta analysis in the Results/Discussion sections of `JAMIA_OpenEMR-AI_Ambient-Notes_v5.docx`.

**Ablation design** (3 variants, all on the same 6 institutional cases):
1. **no-RAG baseline** — schema retrieval disabled; LLM sees only the transcript.
2. **retrieval-depth sweep** — k ∈ {1, 2, 3, 5}.
3. **embedding-model substitution** — all-MiniLM-L6-v2 replaced with ClinicalBERT and PubMedBERT.

Mean ± SD over 3 runs. All other pipeline components held constant.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

HERE = Path('.').resolve()
rag    = pd.read_csv(HERE / 'table2_rag_6case.csv').set_index('model')
norag  = pd.read_csv(HERE / 'table2b_norag_6case.csv').set_index('model')

metric_cols = ['bleu','rouge_l','sbert_coherence','bert_f1',
               'scispacy_entity_recall','medcat_entity_recall']

print('Models:', list(rag.index))

## Table 2 — RAG on (k=2, all-MiniLM-L6-v2)

In [ ]:
rag[metric_cols + ['inference_time_s']]

## Table 2b — no-RAG baseline (schema retrieval disabled)

In [ ]:
norag[metric_cols + ['inference_time_s']]

## Delta analysis — contribution of RAG (Table 2 − Table 2b)

Positive values = RAG improves the metric.

In [ ]:
delta = (rag[metric_cols] - norag[metric_cols]).round(4)
delta

In [ ]:
print('Per-metric drop ranges when RAG is removed (min to max across 7 LLMs):')
for c in metric_cols:
    lo, hi = delta[c].min(), delta[c].max()
    print(f'  {c:28s}  {lo:+.3f}  to  {hi:+.3f}')

## Verification of specific claims in the paper

Each assertion below is one the paper makes in the Results / Discussion text.

In [ ]:
def check(name, expected, actual, tol=1e-3):
    ok = abs(expected - actual) < tol
    print(f"[{'OK' if ok else 'FAIL'}]  {name}: expected {expected}, got {actual:.3f}")

# MedCAT entity recall drops: 0.020 to 0.060 across all seven LLMs
drop_medcat = (rag['medcat_entity_recall'] - norag['medcat_entity_recall']).round(4)
print('MedCAT recall drops per model:')
print(drop_medcat.sort_values(ascending=False))
print(f'  range: {drop_medcat.min():.3f} to {drop_medcat.max():.3f}   (paper: 0.020 to 0.060)')

# Specific MedCAT comparisons
print()
check('MedGemma-27B MedCAT: 0.616 (with RAG)', 0.616, rag.loc['medgemma-27b-4bit','medcat_entity_recall'])
check('MedGemma-27B MedCAT: 0.556 (no RAG)',   0.556, norag.loc['medgemma-27b-4bit','medcat_entity_recall'])
check('GPT-OSS-120B MedCAT: 0.614 (with RAG)', 0.614, rag.loc['gpt-oss-120b','medcat_entity_recall'])
check('GPT-OSS-120B MedCAT: 0.564 (no RAG)',   0.564, norag.loc['gpt-oss-120b','medcat_entity_recall'])
check('Qwen3-32B MedCAT: 0.561 (with RAG)',    0.561, rag.loc['qwen3-32b','medcat_entity_recall'])
check('Qwen3-32B MedCAT: 0.501 (no RAG)',      0.501, norag.loc['qwen3-32b','medcat_entity_recall'])

In [ ]:
# ROUGE-L drops: 0.005 to 0.050, LLaMA-3.1-8B largest (0.269 -> 0.219)
drop_rouge = (rag['rouge_l'] - norag['rouge_l']).round(4)
print('ROUGE-L drops per model:')
print(drop_rouge.sort_values(ascending=False))
print(f'  range: {drop_rouge.min():.3f} to {drop_rouge.max():.3f}   (paper: 0.005 to 0.050)')
print(f'  largest drop: {drop_rouge.idxmax()}   (paper: LLaMA-3.1-8B)')
print()
check('LLaMA-3.1-8B ROUGE-L with RAG',    0.269, rag.loc['llama-3.1-8b','rouge_l'])
check('LLaMA-3.1-8B ROUGE-L no RAG',      0.219, norag.loc['llama-3.1-8b','rouge_l'])

In [ ]:
# BERTScore F1 drops: 0.019 to 0.060; MedGemma-27B 0.871 -> 0.811
drop_bert = (rag['bert_f1'] - norag['bert_f1']).round(4)
print('BERTScore F1 drops per model:')
print(drop_bert.sort_values(ascending=False))
print(f'  range: {drop_bert.min():.3f} to {drop_bert.max():.3f}   (paper: 0.019 to 0.060)')
print()
check('MedGemma-27B BERTScore F1 with RAG', 0.871, rag.loc['medgemma-27b-4bit','bert_f1'])
check('MedGemma-27B BERTScore F1 no RAG',   0.811, norag.loc['medgemma-27b-4bit','bert_f1'])

In [ ]:
# SBERT largest drops: Qwen3-32B 0.507 -> 0.416; MedGemma-4B 0.773 -> 0.693
drop_sbert = (rag['sbert_coherence'] - norag['sbert_coherence']).round(4)
print('SBERT drops per model:')
print(drop_sbert.sort_values(ascending=False))
print()
check('Qwen3-32B SBERT with RAG',   0.507, rag.loc['qwen3-32b','sbert_coherence'])
check('Qwen3-32B SBERT no RAG',     0.416, norag.loc['qwen3-32b','sbert_coherence'])
check('MedGemma-4B SBERT with RAG', 0.773, rag.loc['medgemma-4b','sbert_coherence'])
check('MedGemma-4B SBERT no RAG',   0.693, norag.loc['medgemma-4b','sbert_coherence'])

In [ ]:
# Inference time should be ~unchanged (generation, not retrieval, dominates latency)
dt = (rag['inference_time_s'] - norag['inference_time_s']).round(1)
print('Inference-time deltas (s) — expected ~0:')
print(dt)
print(f'  max |Δt|: {dt.abs().max():.1f}s')

## Retrieval-depth (k) sweep — qualitative

Per the paper's Methods, the k-sweep was run at k ∈ {1, 2, 3, 5} on the same six institutional cases. Findings:

- k=3 and k=5 **substantially degraded** summarization quality vs. k=2.
- Larger k pulled in loosely related SOAP schemas, lowering mean retrieval similarity and causing the LLM to blend organ-system-inappropriate template language.
- Exact-match conditions (Cellulitis, Pneumonia) that dominate at k=1 or k=2 no longer dominated at higher k.
- **Production setting: k=2.**

Per-run k-sweep CSVs were not retained as separate artifacts; the production k=2 run is the `table2_rag_6case.csv` in this directory.

## Embedding-model substitution — qualitative

Substituting all-MiniLM-L6-v2 with ClinicalBERT or PubMedBERT on the same six cases:

- Retrieval-similarity distributions and all downstream summarization metrics fell within run-to-run variance of the all-MiniLM-L6-v2 configuration.
- Retrieval latency increased by **0.92 – 1.32 s per call** with no measurable quality gain.
- **Production embedding: all-MiniLM-L6-v2.**

## FHIR-context ablation — not applicable

The summarization prompt contains only the conversation transcript and retrieved SOAP schemas. FHIR-derived structured resources (Patient, Condition, MedicationRequest, etc.) are used exclusively for clinician display and note write-back — never as input to the generator. There is no FHIR input to remove.

## Reproducing end-to-end

The per-model Modal scripts in `rag_models/models/*_modal.py` can be run with and without schema retrieval. A no-RAG pass is obtained by replacing the `retrieve()` call in `rag_models/pipeline/run_fareez_summaries.py` with an empty schema context (`retrieved_schemas = ''`) and writing to a separate output directory. The k-sweep is obtained by varying the `n_results` argument to `collection.query(...)` in the same file. Embedding-model substitution requires rebuilding the ChromaDB index with `rag_models/pipeline/create_vector_db.py` pointing at ClinicalBERT or PubMedBERT respectively.

The six institutional transcripts, paired OpenEMR extracts, and reference SOAP notes are not included in this repository (they contain institution-specific identifiers from the IU Indianapolis simulation set). The automated metrics above are therefore reproducible from the canonical numeric output (`table2_rag_6case.csv` and `table2b_norag_6case.csv`) even in environments without access to the raw inputs.